# Project Metadata & Overview
*   **File Name:** FoodHub Order Exploratory Data Analysis (EDA) Analysis (Full-Code Submission)
*   **Input File:** foodhub_order.csv
*   **Written By:** Ifrah Khan
*   **Version:** 1.0
*  **Approach:** EDA
*  **Tools & Libraries:** Python (Pandas, Matplotlib, Seaborn), Jupyter Notebook/Google Colab  
*  **Confidentiality:** This dataset is provided as an assignment.

# Introduction
*  **Background:** The number of restaurants in New York is increasing every day, and many busy professionals and students rely on these restaurant meals daily. FoodHub is a food app that connects customers with multiple restaurants, allowing orders to be placed, picked up, and delivered through a centralized platform. FoodHub earns revenue by receiving a small percentage from each food order before giving the rest to the restaurant.
*  **Objective:** The objective is to conduct EDA on FoodHub’s customer order data to gain insights into restaurant demand, customer preferences, and operational performance.

# Import Libraries and Input File

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import Imput File
food_hub_eda = pd.read_csv('/content/drive/MyDrive/foodhub_order.csv', na_values=['Not given'])


ValueError: Mountpoint must not already contain files

# Data Sanity Check
*   **Preliminary Findings 1**: Table 1 and Table 2 show that several observations have "Not given" values in the rating column, which were transformed into NaN (i.e., null values indicating missing data). Table 3 shows that the rating column has 736 missing values, meaning customer ratings are missing for 736 orders.

*    **Preliminary Findings 2**: Table 1 and 2 show that few restaurant names are not consistent, for example, some include information delivery fees or different syntax, requiring standardizing restaurant names.

*    **Preliminary Findings 3**: Table 3 shows that the dataset contains 1898 observations (n=1898), which contains 9 fields in total: 4 integer (order id, customer id, food preparation time, delivery time), 2 float (cost of the order, rating), and 3 categorical (restaurant name, cuisine type, day of the week). Additionally, 1200 customers placed only one order, while the remaining customers placed multiple orders.

In [ ]:
# Data Sanity Check

## Check first 5 rows
print("\nTable 1: Dataset Overview - First 5 Ovservations\n")
display(food_hub_eda.head())

## Check last 5 rows
print("\nTable 2: Dataset Overview - Last 5 Ovservations\n")
display(food_hub_eda.tail())

## Check shape of dataset
print("\nDataset Overview - Dataset Dimensions:\n")
food_hub_eda.shape
display(food_hub_eda.shape)

## Check column information
print("\nTable 3: Dataset Overview - Column Information\n")
col_info = pd.DataFrame({
    "non-null count": food_hub_eda.notnull().sum(),
    "null count": food_hub_eda.isnull().sum(),
    "dtypes": food_hub_eda.dtypes,
    "unique values": food_hub_eda.nunique()
})
col_info = col_info.reset_index().rename(columns={"index": "Column"})
display(col_info)

## Check duplicate rows
print("\nDataset Overview - Duplicate Rows:\n")
print(food_hub_eda.duplicated().sum())

## Check duplicate orders ids
print("\nDataset Overview - Duplicate Order IDs:\n")
print(food_hub_eda.duplicated(subset = 'order_id').sum())

## Check customers who placed at least one order
print("\nDataset Overview - Customers Who Placed at Least One Order:\n")
print(food_hub_eda.duplicated(subset = 'customer_id').sum())

## Check mixed data types across columns
check_col_type = ['order_id', 'customer_id', 'restaurant_name', 'cuisine_type', 'day_of_the_week', 'rating', 'cost_of_the_order', 'food_preparation_time', 'delivery_time']
col_type_summary = {}
for col in check_col_type:
    type_counts = food_hub_eda[col].apply(type).value_counts()
    col_type_summary[col] = {str(k): v for k, v in type_counts.items()}
type_summary = pd.DataFrame(col_type_summary).fillna(0).astype(int).T
print("\nTable 4: Dataset Overview - Check Mixed Data Types Across Columns\n")
display(type_summary)

# Data Preprocessing
*   **Preliminary Findings 4:** Two restaurants were potentially closed (i.e., Empanada Mama) or archived (i.e., Dirty Bird To Go) based on the restaurant name description, and were removed form the dataset since they do not contribute to FoodHub’s current business revenue. As a result, these restaurants were excluded from the analysis, leaving 1882 orders from 176 open businesses.



In [ ]:
# Data Preprocessing

## Clean restaurant name column
def restaurant_name_clean(df, col='restaurant_name'):

    ## Drop closed or archived restaurants
    df = df[~df[col].str.contains("closed|archived", case=False, na=False)].copy()

    ## Create standarized clean column
    df[f"{col}_clean"] = df[col]

    ## Fix text issues and normalize formatting
    df[col + "_clean"] = (df[col].astype(str).replace({"CafÌ© China": "Cafe China", "Joe's Shanghai _Àü£¾÷´": "Joe's Shanghai", "Big Wong Restaurant _¤¾Ñ¼": "Big Wong Restaurant", "'wichcraft": "Wichcraft", "Chipotle Mexican Grill $1.99 Delivery": "Chipotle Mexican Grill"
        })
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
        .str.replace(r"\'S\b", "'s", regex=True)
    )
    return df

## Apply cleaning
food_hub_eda = restaurant_name_clean(food_hub_eda)

## Show unique standarized restaurant names
unique_restaurants = (food_hub_eda[['restaurant_name', 'restaurant_name_clean']].drop_duplicates().reset_index(drop=True))
print(f"\nTable 5: Data Preprocessing – Original Versus Standardized Restaurant Names "f"(N={len(unique_restaurants)})\n")
display(unique_restaurants.style.set_sticky())

## Check shape of dataset
print("\nDataset Overview - Dataset Dimensions:\n")
food_hub_eda.shape
display(food_hub_eda.shape)

# Missing Values

*   **Preliminary Findings 5:** Table 6 shows that a total of 730 orders were missing ratings, representing 38.79% of all orders. Table 7 shows a bivariate analysis of missing ratings by day of the week and restaurant (first 25 restaurants), showing that missing ratings varied across both weekday and weekends.

*   **Preliminary Findings 6:** Figure 1 shows that the Kernal Density Estimate (KDE) of restaurant ratings shows a non-normal distribution, with the most frequent value being a 5-star customer rating. Since there are no extreme outliers, missing ratings were imputed using the central tendency as mode within each single "restaurant-day of the week" group. Table 8 shows that the new rating column (i.e., new_rating) has a total of 65 customer orders with missing ratings.

In [ ]:
# Missing Values

## Univariate analysis in the rate column
df = pd.DataFrame(food_hub_eda)
missing_summary = pd.DataFrame({
    'Count': df.isnull().sum()[df.isnull().sum() > 0],
    'Percentage': (df.isnull().sum()[df.isnull().sum() > 0] / len(df)) * 100
})
print(f"\nTable 6: Missing Values – Univariate Analysis of Ratings\n")
display(missing_summary)

## Bivariate Analysis of Missing Ratings by Day of the Week and Restaurant (First 25 Restaurants)
counts = pd.crosstab(
    df['restaurant_name_clean'],
    df['day_of_the_week'],
    dropna=False,
    margins=False
)
percent = counts.div(counts.sum(axis=1), axis=0).mul(100).round(2).astype(str).add("%")
df = pd.concat({'Count': counts, 'Percent': percent}, axis=1)
print("\nTable 7: Missing Values – Bivariate Analysis of Missing Ratings by Day of the Week and Restaurant (First 25 Restaurants)\n")
df.index.name = "Restaurant"
df.columns.names = ["Metric", "Day of the Week"]
display(df.head(25))

## Kernal Density Estimate: Rating
print("\nFigure 1: Kernal Density Estimate: Rating\n")
plt.figure(figsize=(10, 6))
plt.title('Kernel Density Estimate: Rating')
sns.kdeplot(data=food_hub_eda, x='rating')
plt.xlabel('Rating')
plt.ylabel('Density')
sns.kdeplot(data=food_hub_eda, x='rating')
plt.show()

## Bivariate Analysis of Missing Ratings by Day of the Week and Restaurant (First 25 Restaurants)
food_hub_eda = pd.DataFrame(food_hub_eda)
food_hub_eda['rating_new'] = food_hub_eda['rating'].fillna(
   food_hub_eda.groupby(['restaurant_name_clean', 'day_of_the_week'])['rating']
      .transform(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)
print("\nTable 8: Dataset Overview - Column Information With Imputated Rating Column (i.e., New_Rating)\n")
col_info = pd.DataFrame({
    "non-null count": food_hub_eda.notnull().sum(),
    "null count": food_hub_eda.isnull().sum(),
    "dtypes": food_hub_eda.dtypes,
    "unique values": food_hub_eda.nunique()
})
col_info = col_info.reset_index().rename(columns={"index": "Column"})
display(col_info)

# Outliers

*   **Preliminary Findings 7:** Figure 2 shows 3 boxplots of cost of the order, food preparation time, and delivery time, showing that there are no data points that seem abnormal or unrealistic compared to the rest of the restaurant orders.

In [ ]:
numeric_columns = ['cost_of_the_order', 'food_preparation_time', 'delivery_time']
plt.figure(figsize=(30,6))
print("\nFigure 2: Outlier Analysis - Box Plot of Cost of the Order, Food Preparation Time, Delivery Time\n")
for i, variable in enumerate(numeric_columns, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(x=df[variable], whis=1.5, color="orange")
    plt.tight_layout()
    plt.title(variable.replace("_", " ").title())

# Univariate Analysis

*    **Preliminary Findings 8 (Univariate Table):** Table 7 displays a univariate analysis of the top ten restaurants with the highest number of customer orders, with Shake Shack having the most customer orders (n=219), followed by The Meatball Shop (n=132) and Blue Ribbon Sushi (n=119).

*   **Preliminary Findings 9 (KDE):** Figure 3 shows a bimodal distribution of delivery times, with the highest peak between approximately 25 and 30 minutes and a smaller peak between roughly 15 and 20 minutes. Figure 4 displays a uniform distribution of food preparation times with the highest density between around 20 and 35 minutes. Figure 5 shows a right-skewed distribution of customer order costs, with the highest density between approximately 10 and 15 dollars.

*   **Preliminary Findings 10 (Histogram):** Figure 5 shows that most customer orders cost between about 10 to 15 dollars, with weekend orders being higher in density compared to weekday orders. Figure 6 shows that orders within this price range are American or Japanese cuisine. Figure 7 shows that these orders were more likely to receive 4 or 5 star ratings. Overall, Figures 5-7 show a non-normal, right-skewed distribution, where most order costs are observed on the lower end, while order costs of around 30 dollars or more are much less common. Figure 8 shows that weekend delivery times are more consistently distributed between around 15 to 20 minutes, while weekday deliveries are shown to have longer wait times between about 28 to 32 minutes.

*   **Preliminary Findings 11 (Barplot):** Figure 9 shows a bar plot where American (approximately 410 weekend orders vs. 175 weekday orders) and Japanese cuisine (approximately 340 weekend orders vs. 150 weekday orders) had the highest concentration of customer orders. The cuisines with the lowest customer orders include Thai, Southern, French, Spanish, and Vietnamese, each with about 10 to 15 orders. Figure 10 shows customer ratings were the highest on weekends, with most orders at a 5.0 rating (approximately 780 orders). Figure 11 shows that American and Japanese cuisines were observed with the highest number of customer ratings, typically between 4 to 5 star ratings.

In [ ]:
# Univariate Analysis

## Univariate Analysis - Top 3 Restaurants by Highest Customer Orders
df = pd.DataFrame(food_hub_eda)
top3_restaurants = df['restaurant_name_clean'].value_counts().head(3).reset_index()
top3_restaurants.columns = ['Restaurant', 'Customer Order Count']

print("\nTable 9: Top 3 Restaurants by Highest Customer Orders\n")
display(top3_restaurants)

# Kernal Density Estimate
## Kernal Density Estimate: Delivery Time
print("\nFigure 3: Kernal Density Estimate - Delivery Time\n")
plt.figure(figsize=(10, 6))
plt.title('Kernel Density Estimate: Delivery Time')
sns.kdeplot(data=food_hub_eda, x='delivery_time')
plt.xlabel('Delivery Time')
plt.ylabel('Density')

sns.kdeplot(data=food_hub_eda, x='delivery_time')
plt.show()

## Kernel Density Estimate: Food Preparation Time
print("\nFigure 4: Kernal Density Estimate - Food Preparation Time\n")
plt.figure(figsize=(10, 6))
plt.title('Kernel Density Estimate: Food Preparation Time')
sns.kdeplot(data=food_hub_eda, x='food_preparation_time')
plt.xlabel('Food Preparation Time')
plt.ylabel('Density')

sns.kdeplot(data=food_hub_eda, x='food_preparation_time')
plt.show()

## Kernel Density Estimate: Cost of the Order
print("\nFigure 4: Kernal Density Estimate - Cost of the Order\n")
plt.figure(figsize=(10, 6))
plt.title('Kernel Density Estimate: Cost of the Order')
sns.kdeplot(data=food_hub_eda, x='cost_of_the_order')
plt.xlabel('Cost of the Order ($)')
plt.ylabel('Density')

sns.kdeplot(data=food_hub_eda, x='cost_of_the_order')
plt.show()

# Histogram: Cost of Orders
## Histogram: Cost of Orders by Day of the Week
print("\nFigure 5: Histogram - Cost of the Order by Day of the Week\n")
min_value = food_hub_eda['cost_of_the_order'].min()
max_value = food_hub_eda['cost_of_the_order'].max()
plt.figure(figsize=(10, 6))
plt.title('Histogram: Cost of Orders by Day of the Week')
plt.xlim(min_value, max_value)
plt.xlabel('Cost of the Order ($)')
plt.ylabel('Density')

sns.histplot(data=food_hub_eda, x='cost_of_the_order', hue='day_of_the_week', bins=25, kde=True, stat='density');
plt.show()

## Histogram: Cost of Orders by Cuisine Type
print("\nFigure 6: Histogram - Cost of the Order by Cuisine Type\n")
min_value = food_hub_eda['cost_of_the_order'].min()
max_value = food_hub_eda['cost_of_the_order'].max()
plt.figure(figsize=(10, 6))
plt.title('Histogram: Cost of Orders by Cuisine Type')
plt.xlim(min_value, max_value)
plt.xlabel('Cost of the Order ($)')
plt.ylabel('Density')

sns.histplot(data=food_hub_eda, x='cost_of_the_order', hue='cuisine_type', bins=25, kde=True, stat='density');
plt.show()

## Histogram: Cost of Orders by Rating
print("\nFigure 7: Histogram - Cost of the Order by Rating\n")
min_value = food_hub_eda['cost_of_the_order'].min()
max_value = food_hub_eda['cost_of_the_order'].max()
plt.figure(figsize=(10, 6))
plt.title('Histogram: Cost of Orders by Rating')
plt.xlim(min_value, max_value)
plt.xlabel('Cost of the Order ($)')
plt.ylabel('Density')

sns.histplot(data=food_hub_eda, x='cost_of_the_order', hue='rating_new', bins=25, kde=True, stat='density');
plt.show()

## Histogram: Delivery Time by Day of the Week
print("\nFigure 8: Histogram - Delivery Time by Day of the Week\n")
min_value = food_hub_eda['delivery_time'].min()
max_value = food_hub_eda['delivery_time'].max()
plt.figure(figsize=(10, 6))
plt.title('Histogram: Delivery Time by Day of the Week')
plt.xlim(min_value, max_value)
plt.xlabel('Delivery Time')
plt.ylabel('Density')

sns.histplot(data=food_hub_eda, x='delivery_time', hue='day_of_the_week', bins=25, kde=True, stat='density')
plt.show()

# Barplot
## Barplot: Cuisine Type by Day of the Week
print("\nFigure 9: Barplot - Cuisine Type by Day of the Week\n")
plt.figure(figsize=(10, 6))
sns.countplot(data=food_hub_eda, x='cuisine_type', hue='day_of_the_week')
plt.title('Barplot: Cuisine Type by Day of the Week')
plt.xlabel('Cuisine Type')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

## Barplot: Rating by Day of the Week
print("\nFigure 10: Barplot - Rating by Day of the Week\n")
plt.figure(figsize=(10, 6))
sns.countplot(data=food_hub_eda, x='rating_new', hue='day_of_the_week')
plt.title('Barplot: Rating by Day of the Week')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

## Barplot: Cuisine Type by Rating
print("\nFigure 11: Barplot - Cuisine Type by Rating\n")
plt.figure(figsize=(10, 6))
sns.countplot(data=food_hub_eda, x='cuisine_type', hue='rating_new')
plt.title('Barplot: Cuisine Type by Rating')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Bivariate & Multivariate Analysis

## Heatmap
## Heatmap - Multivariate Matrix of Food Preparation Time vs Cost of the Order vs Delivery Time
print("\nFigure 12: Heatmap - Multivariate Matrix of Food Preparation Time vs. Cost of the Order vs. Delivery Time'\n")
plt.figure(figsize=(10, 6))
sns.heatmap(
    data=food_hub_eda[['food_preparation_time', 'cost_of_the_order', 'delivery_time']].corr(),
    annot=True,
    cmap='YlGnBu',
    vmin=-1,
    vmax=1
)
plt.show()

## Pairplot
## Pairplot - Multivariate Matrix of Food Preparation, Delivery Time, and Order Cost by Day of the Week
print("\nFigure 13: Pairplot - Multivariate Matrix of Food Preparation, Delivery Time, and Order Cost by Day of the Week'\n")
sns.pairplot(
    data=food_hub_eda,
    vars=['food_preparation_time', 'delivery_time', 'cost_of_the_order'],
    hue='day_of_the_week'
)
plt.show()

## Pairplot - Multivariate Matrix of Food Preparation, Delivery Time, and Order Cost by Rating
print("\nFigure 14: Pairplot - Multivariate Matrix of Food Preparation, Delivery Time, and Order Cost by Rating'\n")
sns.pairplot(
    data=food_hub_eda,
    vars=['food_preparation_time', 'delivery_time', 'cost_of_the_order'],
    hue='rating'
)
plt.show()

## Lineplot
## Lineplot - Bivariate Analysis of Food Preparation vs. Cost of the Order by Rating
print("\nFigure 15: Lineplot - Bivariate Analysis of Food Preparation Time vs. Cost of the Order by Rating'\n")
plt.figure(figsize=(10, 6))
df = pd.DataFrame(food_hub_eda)
sns.lineplot(data=df, x='food_preparation_time', y='cost_of_the_order', hue='rating_new', style='rating_new', markers=True, errorbar=('ci', 95))
plt.xlabel('Food Preparation Time')
plt.ylabel('Cost of the Order')
plt.show()

## Lineplot - Bivariate Analysis of Food Preparation vs. Delivery Time by Rating
print("\nFigure 16: Lineplot - Bivariate Analysis of Food Preparation Time vs. Delivery Time by Rating'\n")
plt.figure(figsize=(10, 6))
df = pd.DataFrame(food_hub_eda)
sns.lineplot(data=df, x='food_preparation_time', y='delivery_time', hue='rating_new', style='rating_new', markers=True, errorbar=('ci', 95))
plt.xlabel('Food Preparation Time')
plt.ylabel('Delivery Time')
plt.show()

## Lineplot - Bivariate Analysis of Delivery Time vs. Cost of the Order by Rating
print("\nFigure 17: Lineplot - Bivariate Analysis of Delivery Time vs. Cost of the Order by Rating'\n")
plt.figure(figsize=(10, 6))
df = pd.DataFrame(food_hub_eda)
sns.lineplot(data=df, x='delivery_time', y='cost_of_the_order', hue='rating_new', style='rating_new', markers=True, errorbar=('ci', 95))
plt.xlabel('Delivery Time')
plt.ylabel('Cost of the Order')
plt.show()


## Catplot
## Catplot - Bivariate Analysis of Cost of the Order by Cuisine Type
print("\nFigure 19: Catplot - Bivariate Analysis of Cost of the Order by Cuisine Type\n")
plt.figure(figsize=(30, 6))
df = pd.DataFrame(food_hub_eda)
sns.catplot(data=df, x='cost_of_the_order', col='cuisine_type', col_wrap=7, kind='violin', height=2, aspect=1);
plt.show()

# Bivariate & Multivariate Analysis

*   **Preliminary Findings 12 (Heat map):** Figure 12 shows a multivariate heatmap of food preparation time, cost of the order, and delivery time which have no significiant correlation with each other.

*   **Preliminary Findings 13 (Pair plot):** Figure 13 and Figure 14 show a multivariate pair plot where food preparation time, delivery time, and cost of order are generally consistent across day of the week (i.e., weekday vs. weekend) and customer ratings (i.e., 3-star, 4-star, and 5-star). The scatterplots and KDEs largely overlap, suggesting no major differences or strong relationships among the three variables.

*   **Preliminary Findings 14 (Line plot):** Figures 15, 16, and 17 display bivariate line plots, where customer orders with 5-star ratings show less variability in food preparation time, delivery time, and cost of the order, while 3-star ratings fluctuate more widely across these variables. Figure 18 shows that delivery times are relatively longer on weekdays than on weekends, regardless of food preparation time.

*   **Preliminary Findings 15 (Cat plot):** Figure 19 displays a bivariate cat plot, where American and Japanese cuisines were more likely to have lower costs overall between approximately 10 and 15 dollars. French, Middle Eastern, Indian, and Mediterranean cuisines were the most expensive, with order costs ranging from about 15 to 30 or more dollars.

# Conclusions and Recommendations

*   **Key Findings 1**: Table 1 and 2 show that few restaurant names are not consistent, for example, some include information delivery fees or different syntax, requiring standardizing restaurant names. Additionally, two restaurants were potentially closed (i.e., Empanada Mama) or archived (i.e., Dirty Bird To Go) based on the restaurant name description and were removed from the dataset since they do not contribute to FoodHub’s current business revenue. As a result, these restaurants were excluded from the analysis, leaving 1882 orders from 176 open businesses.
*   **Recommendations 1**: FoodHub app company to implement automated processes to standardize restaurant names, remove closed or archived restaurants, ensuring consistent restaurant naming conventions and analysis on only open businesses that contribute to current business revenue for improved analytics.

*   **Key Findings 2:** Table 6 shows that a total of 730 orders were missing ratings, representing 38.79% of all orders. Table 7 shows that missing ratings varied across both weekday and weekends, while figure 1 shows a non-normal distribution, with the most frequent value being a 5-star customer rating. Since there are no extreme outliers, missing ratings were imputed using the central tendency as mode within each single "restaurant-day of the week" group. Table 8 shows that the new rating column (i.e., new_rating) has a total of 65 customer orders with missing ratings.
*   **Recommendations 2**: FoodHub app company to continue imputing missing ratings using mode within each restaurant–day group, while encouraging customers to leave ratings (e.g., incentives, in-app prompts) to further reduce missing values.

*   **Key Findings 3:** Table 8 shows that 1189 customers placed only one order, while 693 customers placed multiple orders.
*   **Recommendations 3**: FoodHub app company to encourage repeated customers to place multiple orders by offering points, or discounts.

*    **Key Findings 4:** Table 7 displays a univariate analysis of the top ten restaurants with the highest number of customer orders, with Shake Shack having the most customer orders (n=219), followed by The Meatball Shop (n=132) and Blue Ribbon Sushi (n=119). Figure 5 shows that most customer orders cost between about 10 to 15 dollars, with weekend orders being higher in density compared to weekday orders. Figure 6 shows that orders within this price range are American or Japanese cuisine. Figure 7 shows that these orders were more likely to receive 4 or 5 star ratings. Overall, Figures 5-7 show a non-normal, right-skewed distribution, where most order costs are observed on the lower end, while order costs of around 30 dollars or more are much less common. Figure 8 shows that weekend delivery times are more consistently distributed between around 15 to 20 minutes, while weekday deliveries are shown to have longer wait times between about 28 to 32 minutes. Figure 9 shows a bar plot where American (approximately 410 weekend orders vs. 175 weekday orders) and Japanese cuisine (approximately 340 weekend orders vs. 150 weekday orders) had the highest concentration of customer orders. The cuisines with the lowest customer orders include Thai, Southern, French, Spanish, and Vietnamese, each with about 10 to 15 orders. Figure 10 shows customer ratings were the highest on weekends, with most orders at a 5.0 rating (approximately 780 orders). Figure 11 shows that American and Japanese cuisines were observed with the highest number of customer ratings, typically between 4 to 5 star ratings.
*   **Recommendations 4**: FoodHub app company to promote high-demand cuisines (i.e., American, Japanese) with weekend-focused deals, given these cuisines already attract the most orders and high ratings. Additionally, improve weekday delivery times by offering off-peak focused deals to increase revenue during the weekdays.

*   **Key Findings 5:** Figures 15, 16, and 17 display bivariate line plots, where customer orders with 5-star ratings show less variability in food preparation time, delivery time, and cost of the order, while 3-star ratings fluctuate more widely across these variables. Figure 18 shows that delivery times are relatively longer on weekdays than on weekends, regardless of food preparation time. Figure 19 displays a bivariate cat plot, where American and Japanese cuisines were more likely to have lower costs overall between approximately 10 and 15 dollars. French, Middle Eastern, Indian, and Mediterranean cuisines were the most expensive, with order costs ranging from about 15 to 30 or more dollars.
*   **Recommendations 5**: FoodHub app company to improve weekday delivery times by hiring more drivers since delivery delays are more common on the weekdays. Additionally, reduce variability in 3-4 star orders, by setting stricter food preparation targets with reliable delivery drivers to push more orders into the 5-star rating.